# 01 Data Exploration

This notebook starts by loading the wearable body postures dataset from the `SOURCES` folder and inspecting its structure.

## Dataset attributes

- `user` (text)
- `gender` (text)
- `age` (integer)
- `how_tall_in_meters` (real)
- `weight` (integer)
- `body_mass_index` (real)
- `x1` (int, read value of the `x` axis of the 1st accelerometer, mounted on the waist)
- `y1` (int, read value of the `y` axis of the 1st accelerometer, mounted on the waist)
- `z1` (int, read value of the `z` axis of the 1st accelerometer, mounted on the waist)
- `x2` (int, read value of the `x` axis of the 2nd accelerometer, mounted on the left thigh)
- `y2` (int, read value of the `y` axis of the 2nd accelerometer, mounted on the left thigh)
- `z2` (int, read value of the `z` axis of the 2nd accelerometer, mounted on the left thigh)
- `x3` (int, read value of the `x` axis of the 3rd accelerometer, mounted on the right ankle)
- `y3` (int, read value of the `y` axis of the 3rd accelerometer, mounted on the right ankle)
- `z3` (int, read value of the `z` axis of the 3rd accelerometer, mounted on the right ankle)
- `x4` (int, read value of the `x` axis of the 4th accelerometer, mounted on the right upper-arm)
- `y4` (int, read value of the `y` axis of the 4th accelerometer, mounted on the right upper-arm)
- `z4` (int, read value of the `z` axis of the 4th accelerometer, mounted on the right upper-arm)

## Collection protocol note

The dataset was collected by recording one activity at a time in separate blocks (for example: walking for *x* minutes, then standing for *y* minutes, and so on).
Therefore, samples are internally time-ordered within each activity block, but the full natural transition sequence across activities is not preserved.
This means we cannot reliably observe patterns such as `sitting -> standingup -> standing -> sittingdown -> sitting` directly from the row order.

For the `standingup` activity, a person must sit down before standing up again, but these intermediate transitions were either separated into other label blocks or discarded during annotation.

Moreover, because each activity segment can last only around 1-2 seconds in local portions of the signal, it is reasonable to frame (window) consecutive records to obtain a clearer view of the movement dynamics and better reveal short-term motion patterns.

In conclusion, although this dataset supports activity recognition, it does not allow a proper time-series analysis for predicting the next activity from the previous one.

## Target classes

The output labels are:

- `sitting-down`
- `standing-up`
- `standing`
- `walking`
- `sitting`


In [1]:
from pathlib import Path

import pandas as pd

DATASET_FILENAME = 'dataset.csv'
MALFORMED_Z4_REPLACEMENT = "-144"


def project_root() -> Path:
    start = Path.cwd().resolve()
    return start.parents[1]


def dataset_path() -> Path:
    return project_root() / 'SOURCES' / DATASET_FILENAME

def load_exploration_data() -> tuple[pd.DataFrame, Path, pd.Series, pd.DataFrame]:
    data_path = dataset_path()
    df = pd.read_csv(data_path, sep=';', decimal=',', low_memory=False)

    z4_numeric = pd.to_numeric(df['z4'], errors='coerce')
    malformed_z4_mask = df['z4'].notna() & z4_numeric.isna()
    malformed_z4_rows = df.loc[malformed_z4_mask].copy()
    if malformed_z4_mask.any():
        df.loc[malformed_z4_mask, 'z4'] = MALFORMED_Z4_REPLACEMENT

    df['user'] = df['user'].astype('string')
    df['gender'] = df['gender'].astype('category')
    df['class'] = df['class'].astype('category')
    df['z4'] = pd.to_numeric(df['z4'], errors='raise')

    return df, data_path, malformed_z4_mask, malformed_z4_rows

def build_schema(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        [
            ('user', 'text / identifier', df['user'].dtype, 'User identifier'),
            ('gender', 'text / categorical', df['gender'].dtype, 'Participant gender'),
            ('age', 'integer', df['age'].dtype, 'Age in years'),
            ('how_tall_in_meters', 'real', df['how_tall_in_meters'].dtype, 'Height in meters'),
            ('weight', 'integer', df['weight'].dtype, 'Weight in kilograms'),
            ('body_mass_index', 'real', df['body_mass_index'].dtype, 'Body mass index'),
            ('x1', 'integer', df['x1'].dtype, 'Waist accelerometer, x axis'),
            ('y1', 'integer', df['y1'].dtype, 'Waist accelerometer, y axis'),
            ('z1', 'integer', df['z1'].dtype, 'Waist accelerometer, z axis'),
            ('x2', 'integer', df['x2'].dtype, 'Left thigh accelerometer, x axis'),
            ('y2', 'integer', df['y2'].dtype, 'Left thigh accelerometer, y axis'),
            ('z2', 'integer', df['z2'].dtype, 'Left thigh accelerometer, z axis'),
            ('x3', 'integer', df['x3'].dtype, 'Right ankle accelerometer, x axis'),
            ('y3', 'integer', df['y3'].dtype, 'Right ankle accelerometer, y axis'),
            ('z3', 'integer', df['z3'].dtype, 'Right ankle accelerometer, z axis'),
            ('x4', 'integer', df['x4'].dtype, 'Right upper-arm accelerometer, x axis'),
            ('y4', 'integer', df['y4'].dtype, 'Right upper-arm accelerometer, y axis'),
            ('z4', 'integer', df['z4'].dtype, 'Right upper-arm accelerometer, z axis'),
            ('class', 'categorical / target', df['class'].dtype, 'Activity label'),
        ],
        columns=['attribute', 'semantic_type', 'pandas_dtype', 'description'],
    )


df, data_path, malformed_z4_mask, malformed_z4_rows = load_exploration_data()

output_path = project_root() / 'NOTEBOOKS' / 'DATASET' / 'baseline.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print(f'Loaded dataset from: {data_path.resolve()}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'Wrote corrected dataset to: {output_path.resolve()}')

print('Malformed z4 row:')
print(f'Count: {int(malformed_z4_mask.sum()):,}')
if malformed_z4_mask.any():
    display(malformed_z4_rows)
    print('Replaced the malformed z4 value with -144.')

print('First rows:')
display(df.head())

print('Random sample:')
display(df.sample(5, random_state=42))

print('Column names:')
print(list(df.columns))

print('Schema:')
display(build_schema(df))

Loaded dataset from: C:\Users\matte\Desktop\DMML_Wearable_Body_Postures\SOURCES\dataset.csv
Shape: 165,633 rows x 19 columns
Wrote corrected dataset to: C:\Users\matte\Desktop\DMML_Wearable_Body_Postures\NOTEBOOKS\DATASET\baseline.csv
Malformed z4 row:
Count: 1


,user,gender,age,how_tall_in_meters,weight,body_mass_index,x1,y1,z1,x2,y2,z2,x3,y3,z3,x4,y4,z4,class
122076,jose_carlos,Man,75,1.67,67,24.0,-8,101,-120,-13,91,-101,17,123,-108,-207,-82,-14420-11-2011 04:50:23.713,standingup


Replaced the malformed z4 value with -144.
First rows:


,user,gender,age,how_tall_in_meters,weight,body_mass_index,x1,y1,z1,x2,y2,z2,x3,y3,z3,x4,y4,z4,class
0,debora,Woman,46,1.62,75,28.6,-3,92,-63,-23,18,-19,5,104,-92,-150,-103,-147,sitting
1,debora,Woman,46,1.62,75,28.6,-3,94,-64,-21,18,-18,-14,104,-90,-149,-104,-145,sitting
2,debora,Woman,46,1.62,75,28.6,-1,97,-61,-12,20,-15,-13,104,-90,-151,-104,-144,sitting
3,debora,Woman,46,1.62,75,28.6,-2,96,-57,-15,21,-16,-13,104,-89,-153,-103,-142,sitting
4,debora,Woman,46,1.62,75,28.6,-1,96,-61,-13,20,-15,-13,104,-89,-153,-104,-143,sitting


Random sample:


,user,gender,age,how_tall_in_meters,weight,body_mass_index,x1,y1,z1,x2,y2,z2,x3,y3,z3,x4,y4,z4,class
129314,debora,Woman,46,1.62,75,28.6,-1,92,-94,8,79,-104,31,100,-98,-172,-96,-147,walking
14854,debora,Woman,46,1.62,75,28.6,-5,66,-55,-10,14,-37,-1,94,-72,-160,-106,-171,sitting
104035,wallace,Man,31,1.71,83,28.4,-7,96,-122,6,91,-137,35,125,-77,-179,-69,-153,standing
31651,wallace,Man,31,1.71,83,28.4,4,71,-59,-25,4,-22,92,84,-137,-97,-118,-191,sitting
31542,wallace,Man,31,1.71,83,28.4,1,70,-63,-27,1,-26,95,88,-134,-93,-113,-183,sitting


Column names:
['user', 'gender', 'age', 'how_tall_in_meters', 'weight', 'body_mass_index', 'x1', 'y1', 'z1', 'x2', 'y2', 'z2', 'x3', 'y3', 'z3', 'x4', 'y4', 'z4', 'class']
Schema:


,attribute,semantic_type,pandas_dtype,description
0,user,text / identifier,string,User identifier
1,gender,text / categorical,category,Participant gender
2,age,integer,int64,Age in years
3,how_tall_in_meters,real,float64,Height in meters
4,weight,integer,int64,Weight in kilograms
5,body_mass_index,real,float64,Body mass index
6,x1,integer,int64,"Waist accelerometer, x axis"
7,y1,integer,int64,"Waist accelerometer, y axis"
8,z1,integer,int64,"Waist accelerometer, z axis"
9,x2,integer,int64,"Left thigh accelerometer, x axis"


## Missing Values Check

This check is kept in a dedicated cell because missing-value handling is a preprocessing decision.
We only inspect the missingness pattern here; we do not fill or delete values now, because that would be done later inside the training pipeline to avoid data leakage.

In [2]:
missing_values = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percentage': (df.isna().mean() * 100).round(4),
}).sort_values(['missing_count', 'missing_percentage'], ascending=False)

print('Missing values by column:')
display(missing_values)

print('\nDecision for later preprocessing:')
print('- If missing values are found in the training data, handle them inside the pipeline.')
print('- Do not fill or delete them here, because that would risk data leakage.')

Missing values by column:


,missing_count,missing_percentage
user,0,0.0
gender,0,0.0
age,0,0.0
how_tall_in_meters,0,0.0
weight,0,0.0
body_mass_index,0,0.0
x1,0,0.0
y1,0,0.0
z1,0,0.0
x2,0,0.0



Decision for later preprocessing:
- If missing values are found in the training data, handle them inside the pipeline.
- Do not fill or delete them here, because that would risk data leakage.


Augmented Dataset with
- magnitude
- euler angles: roll, yaw, pitch

In [3]:
import numpy as np

ACCELEROMETER_FEATURES = ['x1', 'y1', 'z1', 'x2', 'y2', 'z2', 'x3', 'y3', 'z3', 'x4', 'y4', 'z4']
ACCELEROMETER_MAGNITUDE_FEATURES = ['mag1', 'mag2', 'mag3', 'mag4']
ACCELEROMETER_ANGLE_FEATURES = ['roll1', 'pitch1', 'yaw1', 'roll2', 'pitch2', 'yaw2', 'roll3', 'pitch3', 'yaw3', 'roll4', 'pitch4', 'yaw4']

def add_accelerometer_magnitudes_and_angles(df):
    missing_features = [feature for feature in ACCELEROMETER_FEATURES if feature not in df.columns]
    if missing_features:
        raise ValueError(f'Missing accelerometer features: {missing_features}')

    transformed = df.copy()
    for sensor_id in range(1, 5):
        x_col, y_col, z_col = f'x{sensor_id}', f'y{sensor_id}', f'z{sensor_id}'
        x_values = transformed[x_col].astype(float)
        y_values = transformed[y_col].astype(float)
        z_values = transformed[z_col].astype(float)
        transformed[f'mag{sensor_id}'] = np.sqrt(x_values ** 2 + y_values ** 2 + z_values ** 2)
        transformed[f'roll{sensor_id}'] = np.degrees(np.arctan2(y_values, z_values))
        transformed[f'pitch{sensor_id}'] = np.degrees(np.arctan2(-x_values, np.sqrt(y_values ** 2 + z_values ** 2)))
        transformed[f'yaw{sensor_id}'] = np.degrees(np.arctan2(y_values, x_values))
    return transformed

# add to the dataset df magnitude and euler angles and store it in project_root()/DATASET/augmented.csv
df_augmented = add_accelerometer_magnitudes_and_angles(df)
augmented_output_path = project_root() / 'NOTEBOOKS' / 'DATASET' / 'augmented.csv'
df_augmented.to_csv(augmented_output_path, index=False)
print(f'Augmented dataset with magnitudes and angles, and wrote to: {augmented_output_path.resolve()}')

Augmented dataset with magnitudes and angles, and wrote to: C:\Users\matte\Desktop\DMML_Wearable_Body_Postures\NOTEBOOKS\DATASET\augmented.csv
